# Exploration de l'API OpenWeather
Le but de ce notebook est d'explorer les données envoyées par l'API d'OpenWeather dans le but de voir la forme des données, de chercher ce qu'elles contiennent, de vérifier ce qui est pertinent à garder, de voir comment les nettoyer et de transformer en un schéma SQL.

## Connexion à l'API Current Weather
Je teste la connexion à l'API qui renvoie les informations météo d'une ville. Et, aussi une API d'OpenWeather qui renvoie les coordonnées géographiques d'une ville. La clé est hardcodée uniquement pour exploration, jamais en production

In [3]:
#Importation des librairies nécessaires
import requests

In [1]:
cle_api = '1c93bb26f3be803708c12726e1a23b72'

In [4]:
# Appel API pour les coordonnées de les ville utilisées en exemple
# Libreville
params_lib = {
    'q': 'Libreville,GA',
    'appid': cle_api
}
url = "http://api.openweathermap.org/geo/1.0/direct"

r_lib = requests.get(url, params=params_lib)

geo_lib = [r_lib.json()[0]['lat'], r_lib.json()[0]['lon']]

# Paris
params_pa = {
    'q': 'Paris,FR',
    'appid': cle_api
}
url = "http://api.openweathermap.org/geo/1.0/direct"

r_pa = requests.get(url, params=params_pa)

geo_pa = [r_pa.json()[0]['lat'], r_pa.json()[0]['lon']]

# Tokyo
params_to = {
    'q': 'Tokyo,JP',
    'appid': cle_api
}
url = "http://api.openweathermap.org/geo/1.0/direct"

r_to = requests.get(url, params=params_to)

geo_to = [r_to.json()[0]['lat'], r_to.json()[0]['lon']]

# Reykjavík
params_rey = {
    'q': 'Reykjavík',
    'appid': cle_api
}
url = "http://api.openweathermap.org/geo/1.0/direct"

r_rey = requests.get(url, params=params_rey)

geo_rey = [r_rey.json()[0]['lat'], r_rey.json()[0]['lon']]

In [5]:
# Appel de l'API pour les informations météo d'une ville donnée
url = "https://api.openweathermap.org/data/2.5/weather"

# Libreville
p_weather_lib = {
    'lat': geo_lib[0],
    'lon': geo_lib[1],
    'appid': cle_api
}

r_lib = requests.get(url, params=p_weather_lib)

# Paris
p_weather_pa = {
    'lat': geo_pa[0],
    'lon': geo_pa[1],
    'appid': cle_api
}

r_pa = requests.get(url, params=p_weather_pa)

# Tokyo
p_weather_to = {
    'lat': geo_to[0],
    'lon': geo_to[1],
    'appid': cle_api
}

r_to = requests.get(url, params=p_weather_to)

# Reykjavík
p_weather_rey = {
    'lat': geo_rey[0],
    'lon': geo_rey[1],
    'appid': cle_api
}

r_rey = requests.get(url, params=p_weather_rey)

In [6]:
# Libreville
d_lib = r_lib.json()

In [7]:
# Paris
d_pa = r_pa.json()

In [8]:
# Tokyo
d_to = r_to.json()

In [9]:
# Reykjavík
d_rey = r_rey.json()

## Analyse des réponses API
Ici, on va récupérer les clés et vérifier si toutes sont présentes dans chaque réponse.
Déjà, on peut déjà juger que les données sont présentées sous le format clé-valeur. Les valeurs sont soit imbriqué, soit des listes, soit juste une valeur.

In [10]:
d_lib.keys()

dict_keys(['coord', 'weather', 'base', 'main', 'visibility', 'wind', 'clouds', 'dt', 'sys', 'timezone', 'id', 'name', 'cod'])

In [11]:
d_pa.keys()

dict_keys(['coord', 'weather', 'base', 'main', 'visibility', 'wind', 'clouds', 'dt', 'sys', 'timezone', 'id', 'name', 'cod'])

In [12]:
d_to.keys()

dict_keys(['coord', 'weather', 'base', 'main', 'visibility', 'wind', 'clouds', 'dt', 'sys', 'timezone', 'id', 'name', 'cod'])

In [13]:
d_rey.keys()

dict_keys(['coord', 'weather', 'base', 'main', 'visibility', 'wind', 'clouds', 'dt', 'sys', 'timezone', 'id', 'name', 'cod'])

On remarque que les clés sont les mêmes pour les quatre villes. On peut en déduire que ce sont des informations qui sont obligatoirement transmises. Analysons les clés ayant des valeurs imbriquées. On va analyser : weather, main, wind, clouds et sys.

In [14]:
print("La clé 'weather' :\n ", d_lib['weather'])
print("La clé 'main' :\n ", d_lib['main'])
print("La clé 'wind' :\n ", d_lib['wind'])
print("La clé 'clouds' :\n ", d_lib['clouds'])
print("La clé 'sys' :\n ", d_lib['sys'])

La clé 'weather' :
  [{'id': 801, 'main': 'Clouds', 'description': 'few clouds', 'icon': '02d'}]
La clé 'main' :
  {'temp': 298.16, 'feels_like': 298.75, 'temp_min': 298.16, 'temp_max': 298.16, 'pressure': 1009, 'humidity': 78, 'sea_level': 1009, 'grnd_level': 1008}
La clé 'wind' :
  {'speed': 1.54, 'deg': 0}
La clé 'clouds' :
  {'all': 20}
La clé 'sys' :
  {'type': 1, 'id': 2190, 'country': 'GA', 'sunrise': 1770183184, 'sunset': 1770226745}


Ainsi, d'après nos observations, on en conclut que :
- "weather" contient une liste de clé-valeur dont les clés sont id, main, description, et icon;
- "main" contient un dictionnaire avec les clés étant temp, feels_like, temp_min, temp_max, pressure, humidity, sea_level et grnd_level;
- "wind" a pour valeur un dictionnaire avec speed et deg comme clés;
- "clouds", sa valeur est aussi un dictionnaire, mais ici, elle a seulement la clé all;
- "sys" contient un dictionnaire avec les clés type, id, country, sunrise, et sunset.

Les champs cités sont toujours présents pour les villes utilisées.
Maintenant, quelles informations seront utiles pour une office de tourisme ? 
Voici les clés qui seront gardées : coord (lat, lon), weather (main, description), main (temp, temp_min, temp_max, pressure, humidity), visibility, wind (speed), clouds, dt, sys (country, sunrise, sunset), timezone, name (à ne pas utiliser tel quel selon les villes).
Voici les clés qui seront jetées : weather (id, icon), base, main (sea_level, grnd_level), wind (deg), sys (type, id), id, cod (on vérifiera le code renvoyé, mais on ne garde pas en mémoire).

En conclusion, le schema SQL sera le suivant :

Meteo
- main,
- description,
- temp,
- feels_like
- temp_min,
- temp_max,
- pressure,
- humidity,
- visibility
- wind_speed,
- clouds
- date
- sunrise
- sunset

Localisation
- country_name
- country_code
- city
- timezone
- lat
- lon


Les deux tables seront reliées via une relation Meto N-1 Localisation.